# Reactive covering: measured results and audit

README.md derives the models; RESULTS.md documents certificates and seeded timing. Budget points use full passes and priorities, not a complete frontier.

In [ ]:
from pathlib import Path
import json,pandas as pd
import matplotlib.pyplot as plt
from solution import booking,evaluate
s=pd.read_csv('results/summary.csv')
display(s)
display(s[s.budget.isna()].groupby(['dataset','method','partial']).agg(cost=('cost','mean'),time=('elapsed','mean')))

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,4),constrained_layout=True)
for ax,(name,g) in zip(axes,s.groupby('dataset',sort=False)):
    for (method,partial),r in g.groupby(['method','partial']):
        ax.scatter(r.cost,r.satisfied/r.total,label=f'{method}; partial={partial}')
    ax.set(title=name,xlabel='Cost',ylabel='Satisfied fraction');ax.grid(alpha=.2)
axes[-1].legend(fontsize=6);plt.show()

## Reconstruct, independently validate and export every recommended plan

In [ ]:
for name,triples in json.loads(Path('results/intervals.json').read_text()).items():
    d=json.loads(Path(f'data/{name}.json').read_text())
    sol={'Bookings':[booking(d,p,a,b) for p,a,b in triples]}
    r=evaluate(d,sol)
    assert r['satisfied']==r['total'] and not r['unreachable_ids']
    print(name,r['cost'],r['satisfied'])
    Path(f'{name}_optimized.json').write_text(json.dumps(sol,indent=2))

Run `python -m pytest -q` and `python experiment.py` to reproduce. Nonzero gaps and missing bounds are not optimality certificates. Short seeded runtime excludes discovery. See slides.html for the presentation.